# Quantum ML Lab — Publication Analysis

Loads experiment logs via `dashboard.benchmark_data`, builds the publication
leaderboard (multi-seed mean ± bootstrap CI), plots figures with the shared
`plot_style` module, and surfaces applicability gates / negative results.

**Prerequisites:** run at least one experiment so `logs/experiments.jsonl` exists.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dashboard.benchmark_data import (
    load_applicability_gates,
    load_records,
    to_leaderboard_rows,
)
from src.training.plot_style import apply_publication_style

apply_publication_style()
records = load_records()
leaderboard = to_leaderboard_rows(records)
gates = load_applicability_gates(records)

print(f"{len(records)} log records | {len(leaderboard)} leaderboard rows | {len(gates)} gates")

In [ ]:
if leaderboard:
    df = pd.DataFrame(leaderboard)
    cols = ["exp_id", "model", "accuracy", "ci_low", "ci_high", "n_seeds", "source"]
    display(df[cols].sort_values(["exp_id", "accuracy"], ascending=[True, False]))

    # Best model per experiment
    best = df.loc[df.groupby("exp_id")["accuracy"].idxmax()]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(best["exp_id"], best["accuracy"], color="#009E73")
    ax.set_ylabel("Holdout accuracy (%)")
    ax.set_title("Best model per experiment (publication leaderboard)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No leaderboard data. Run experiments first: python experiments/exp_001_quantum_vs_classical/run.py")

In [ ]:
# Applicability gates and documented negative results
if gates:
    gate_df = pd.DataFrame(gates)
    display(gate_df.sort_values("exp_id"))
else:
    print("No applicability gates logged yet.")

print("\nNegative results documented in docs/negative_results.md:")
print("  exp_005 curriculum, exp_007 self-play, exp_003/009 entanglement")